In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
data = pd.read_csv('data.csv')

data.head()

,DateTime,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption
0,1/1/2017 0:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386
1,1/1/2017 0:10,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434
2,1/1/2017 0:20,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373
3,1/1/2017 0:30,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711
4,1/1/2017 0:40,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964


In [4]:
data['DateTime'] = pd.to_datetime(data['DateTime'])
data['DateTime'] = pd.to_datetime(data['DateTime'])
data['date'] = data['DateTime'].dt.date
data['time'] = data['DateTime'].dt.time

data.head()

,DateTime,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption,date,time
0,2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,2017-01-01,00:00:00
1,2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,2017-01-01,00:10:00
2,2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,2017-01-01,00:20:00
3,2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,2017-01-01,00:30:00
4,2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,2017-01-01,00:40:00


In [9]:
data['hour'] = data['DateTime'].dt.hour

data.head(10)

,DateTime,Temperature,Humidity,Wind Speed,general diffuse flows,diffuse flows,Zone 1 Power Consumption,Zone 2 Power Consumption,Zone 3 Power Consumption,date,time,hour
0,2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,2017-01-01,00:00:00,0
1,2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,2017-01-01,00:10:00,0
2,2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,2017-01-01,00:20:00,0
3,2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,2017-01-01,00:30:00,0
4,2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,2017-01-01,00:40:00,0
5,2017-01-01 00:50:00,5.853,76.9,0.081,0.059,0.108,26624.81013,17416.41337,18130.12048,2017-01-01,00:50:00,0
6,2017-01-01 01:00:00,5.641,77.7,0.080,0.048,0.096,25998.98734,16993.31307,17945.06024,2017-01-01,01:00:00,1
7,2017-01-01 01:10:00,5.496,78.2,0.085,0.055,0.093,25446.07595,16661.39818,17459.27711,2017-01-01,01:10:00,1
8,2017-01-01 01:20:00,5.678,78.1,0.081,0.066,0.141,24777.72152,16227.35562,17025.54217,2017-01-01,01:20:00,1
9,2017-01-01 01:30:00,5.491,77.3,0.082,0.062,0.111,24279.49367,15939.20973,16794.21687,2017-01-01,01:30:00,1


In [13]:
print(data.shape)

feature_data = data[['Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows', 'hour']]
target_data = data[['Zone 1 Power Consumption']]

print(feature_data.shape)

(52416, 12)
(52416, 6)


In [23]:
    split_point = int(len(feature_data) * 0.75)

    # Train 75%
    train_features = feature_data[:split_point]
    train_target = target_data[:split_point]

    # Test 25%
    test_features = feature_data[split_point:]
    test_target = target_data[split_point:]

In [25]:
def sliding_window(feature_data, target_data, window_size=144, pred_horizon=24, stride=6):

    X = []
    y = []

    position = window_size

    while position + pred_horizon <= len(feature_data):
        window = feature_data[position - window_size : position]

        targets = target_data[position : position + pred_horizon].values.flatten()

        flattened = window.values.flatten()

        X.append(flattened)
        y.append(targets)

        position = position + stride

    return np.array(X), np.array(y)


In [27]:
X_train, y_train = sliding_window(train_features, train_target)
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

X_test, y_test = sliding_window(test_features, test_target)
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (6525, 864), y_train: (6525, 24)
X_test: (2157, 864), y_test: (2157, 24)
